Run these examples from the complete `tcia_utils_v4` folder; see [setup and preview-release instructions](README.md).

# Participants, clinical scope, and imaging availability

Participant identity is scoped to a TCIA dataset. Clinical subject IDs may need a reviewed crosswalk before joining to inventory IDs. A dataset-level inferred diagnosis is not an observed patient fact. This notebook reads small metadata selections only; live routes that fail are reported as unavailable.

In [ ]:
from importlib.metadata import version
from _support import MODE, SHORT_TITLE, DOI, make_client, checked
print("SYNTHETIC FIXTURE MODE — results below are examples, not live TCIA evidence" if MODE == "fixture"
      else "LIVE METADATA MODE — bounded calls to the configured TCIA service")
print("tcia_utils", version("tcia_utils"))

In [ ]:
with make_client() as tcia:
    participants = checked("participant search", lambda: tcia.search_participants(
        short_titles=[SHORT_TITLE], limit=1))
    if participants:
        for row in participants.items:
            print("participant", row.get("dataset_type"), row.get("short_title"), row.get("participant_key"))
        print("identity scope", participants.response.get("identity_scope"))
        participant_key = participants.items[0]["participant_key"] if participants.items else None
    else:
        participant_key = None
    if participant_key:
        detail = checked("participant detail", lambda: tcia.get_participant(participant_key))
        if detail:
            print("source identifiers", len(detail.data["identifiers"]))
        assets = checked("participant assets", lambda: tcia.list_participant_assets(participant_key, limit=2))
        if assets:
            print("available asset summaries", assets.count, "completeness", assets.completeness.state)

In [ ]:
with make_client() as tcia:
    subjects = checked("clinical subjects", lambda: tcia.list_clinical_subjects(SHORT_TITLE, limit=50))
    if subjects:
        for row in subjects.data["subjects"][:2]:
            print("clinical subject", row.get("subject_id"), "diagnosis", row.get("primary_diagnosis"),
                  "inferred", row.get("primary_diagnosis_is_inferred"))
        print("clinical list completeness", subjects.completeness.state)
    facts = checked("clinical facts", lambda: tcia.list_clinical_facts(SHORT_TITLE, limit=100))
    if facts:
        for row in facts.data["facts"][:2]:
            print("fact", row.get("subject_id"), row.get("concept"),
                  "scope", row.get("evidence_scope"), "inferred", row.get("is_inferred"))

The bounded clinical lists are not full-cohort exports. `coverage_complete` only describes participant linking. Keep source priority, conflicts, and inference flags when analyzing clinical evidence; do not infer a patient diagnosis from a dataset label.